# PINN Demo (MECHANISM-AI)

This notebook demonstrates a simple Physics-Informed Neural Network (PINN) learning dynamics from a differential equation.

Equation: dx/dt = -x

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [ ]:
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, t):
        return self.net(t)

In [ ]:
def pinn_loss(model, t):
    t.requires_grad_(True)
    x = model(t)

    dx_dt = torch.autograd.grad(
        x, t,
        grad_outputs=torch.ones_like(x),
        create_graph=True
    )[0]

    residual = dx_dt + x
    return torch.mean(residual**2)

In [ ]:
model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

t = torch.linspace(0, 1, 100).view(-1, 1)

for epoch in range(500):
    optimizer.zero_grad()
    loss = pinn_loss(model, t)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

In [ ]:
t_test = torch.linspace(0, 1, 100).view(-1, 1)

with torch.no_grad():
    x_pred = model(t_test)

plt.plot(t_test.numpy(), x_pred.numpy(), label='PINN solution')
plt.xlabel('t')
plt.ylabel('x(t)')
plt.title('PINN Approximation of dx/dt = -x')
plt.legend()
plt.show()